# NLLB-200 Translation Notebook (RTX 2050 Optimized)
This notebook translates English↔Sinhala using Meta's NLLB-200 with auto-save, resume, and OOM recovery.

In [1]:
%pip install -q torch transformers sentencepiece pandas tqdm


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
"""
NLLB Translation V3 - Optimized for RTX 2050 4GB
"""

import gc
import logging
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ---------------- CONFIG ----------------
MODEL_NAME = "facebook/nllb-200-distilled-600M"

INPUT_CSV = "../Pre processed Data/sold_train_clean.csv"
OUTPUT_CSV = "../Translate/sold_train_translation.csv"

TEXT_COLUMN = "text"
LANG_COLUMN = "lang"
TRANS_COLUMN = "text_trans"

MAX_LENGTH = 128
INITIAL_BATCH_SIZE = 8
MIN_BATCH_SIZE = 1
SAVE_EVERY = 1000

logging.basicConfig(
    filename="translation.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("="*60)
print("NLLB Translation V3")
print("="*60)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB")

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model (first run may download ~1.5GB)...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
model.to(device)
model.eval()
print("Model loaded.\n")

# -------- dataset --------
if Path(OUTPUT_CSV).exists():
    print("Resuming from existing output...")
    df = pd.read_csv(OUTPUT_CSV)
else:
    df = pd.read_csv(INPUT_CSV)

if TRANS_COLUMN not in df.columns:
    df[TRANS_COLUMN] = ""

df[TRANS_COLUMN] = df[TRANS_COLUMN].fillna("").astype(str)
df[LANG_COLUMN] = df[LANG_COLUMN].astype(str).str.lower().str.strip()

print("Already translated:", (df[TRANS_COLUMN].str.strip()!="").sum())
print("Remaining:", (df[TRANS_COLUMN].str.strip()=="").sum())

def save():
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUTPUT_CSV,index=False,encoding="utf-8-sig")

def translate_batch(texts, src, tgt):
    tokenizer.src_lang = src
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH
    )
    inputs = {k:v.to(device) for k,v in inputs.items()}
    with torch.no_grad():
        out = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt),
            do_sample=False,
            num_beams=1,
            max_length=MAX_LENGTH
        )
    res = tokenizer.batch_decode(out, skip_special_tokens=True)
    del inputs, out
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return res

translated_since_save=0

for lang, src, tgt in [("en","eng_Latn","sin_Sinh"),("si","sin_Sinh","eng_Latn")]:
    idx = df[(df[LANG_COLUMN]==lang) & (df[TRANS_COLUMN].str.strip()=="")].index.tolist()
    if not idx:
        continue
    print(f"\n{src} -> {tgt} : {len(idx)} rows")
    batch_size = INITIAL_BATCH_SIZE
    pbar = tqdm(total=len(idx), desc=f"{src}->{tgt}", unit="rows")
    pos = 0
    while pos < len(idx):
        batch_idx = idx[pos:pos+batch_size]
        texts = df.loc[batch_idx,TEXT_COLUMN].fillna("").astype(str).tolist()
        try:
            result = translate_batch(texts, src, tgt)
            df.loc[batch_idx,TRANS_COLUMN] = result
            pos += len(batch_idx)
            translated_since_save += len(batch_idx)
            pbar.update(len(batch_idx))
            pbar.set_postfix(batch=batch_size)
            if translated_since_save >= SAVE_EVERY:
                save()
                logging.info("Auto-saved")
                print(f"\nAuto-saved after {translated_since_save} rows.")
                translated_since_save = 0
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                torch.cuda.empty_cache()
                gc.collect()
                if batch_size > MIN_BATCH_SIZE:
                    batch_size = max(MIN_BATCH_SIZE, batch_size//2)
                    print(f"\nOOM detected. Retrying with batch size {batch_size}")
                    continue
                else:
                    print("Skipping one problematic row.")
                    logging.exception(e)
                    pos += 1
                    pbar.update(1)
            else:
                raise
    pbar.close()

save()
logging.info("Completed")
print("\nTranslation complete.")
print("Output:", OUTPUT_CSV)
print("Log:", "translation.log")


from IPython.display import display
print('\nPreview of translated data:')
display(df.head())
print(f'Total rows: {len(df)}')


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NLLB Translation V3
Device: cuda
GPU : NVIDIA GeForce RTX 2050
VRAM: 4.00 GB

Loading tokenizer...
Loading model (first run may download ~1.5GB)...
Model loaded.

Already translated: 0
Remaining: 7500

sin_Sinh -> eng_Latn : 7500 rows


sin_Sinh->eng_Latn:  13%|█▎        | 1000/7500 [04:04<29:18,  3.70rows/s, batch=8]


Auto-saved after 1000 rows.


sin_Sinh->eng_Latn:  27%|██▋       | 2000/7500 [08:05<15:05,  6.07rows/s, batch=8]

...


Auto-saved after 1000 rows.


sin_Sinh->eng_Latn: 100%|██████████| 7500/7500 [33:46<00:00,  3.70rows/s, batch=8]


Translation complete.
Output: ../Translate/sold_train_translation.csv
Log: translation.log

Preview of translated data:


,text,label,lang,text_trans
0,පට්ට පට පට...,0,si,It's a great film.
1,පරණ කෑල්ල අද වෙනකම් හිටියනම් අදට අවුරුදු 4යි. ...,1,si,"If the old one had stayed up to date, it'd be ..."
2,යාළුවා කියලා හිතන් සර් ගේ ඔලුවට රෙද්ද දාලා නෙල...,0,si,It's a lot of fun to get a friend on the head ...
3,හොඳ මිතුරියක් කතා කලා. විස්තර කතාකරමින් ඉදලා ම...,1,si,"A good friend called me, and after talking abo..."
4,"ඔය බනින්නෙ.. හරකා, මී හරකා කිය කිය...",1,si,"You're the one who blames the bull, the bee, t..."


Total rows: 7500
